# OpticDisc-DomainGuard — clean reproduction
This notebook orchestrates the public modular implementation. It is **not** represented as the untouched historical Colab notebook.

Scientific rule: fixed external transfer uses HYGD for fitting/model selection/threshold selection; ACRIMA and G1020 labels are evaluation-only until an experiment is explicitly declared target adaptation.

In [ ]:
!git clone https://github.com/eyasudesalegne/OpticDisc-DomainGuard.git
%cd OpticDisc-DomainGuard
!pip -q install -r requirements.txt

## 1. Provide the manifest
Create `data/manifests/domainguard_manifest.csv` following `data/README.md`. Raw datasets are not redistributed.

In [ ]:
from pathlib import Path
manifest = Path('data/manifests/domainguard_manifest.csv')
assert manifest.exists(), 'Create the manifest first; see data/README.md'


## 2. Extract frozen ResNet-50 features

In [ ]:
!python scripts/02_extract_features.py --manifest data/manifests/domainguard_manifest.csv --backbone resnet50 --branch raw_resize --out results/features/resnet50_raw.csv

## 3. Grouped HYGD internal validation

In [ ]:
!python scripts/03_internal_validation.py --features results/features/resnet50_raw.csv --folds 10

## 4. Fixed external transfer

In [ ]:
!python scripts/04_external_transfer.py --features results/features/resnet50_raw.csv --source HYGD --targets ACRIMA G1020

## 5. Dataset-identity audit

In [ ]:
!python scripts/05_domain_identity_audit.py --features results/features/resnet50_raw.csv

## 6. Few-shot target adaptation (separate from zero-shot transfer)

In [ ]:
!python scripts/07_fewshot_adaptation.py --features results/features/resnet50_raw.csv --target ACRIMA --out results/metrics/fewshot_acrima.csv
!python scripts/07_fewshot_adaptation.py --features results/features/resnet50_raw.csv --target G1020 --out results/metrics/fewshot_g1020.csv

## Modern frozen backbones
Repeat feature extraction with `efficientnet_b0`, `convnext_tiny`, and `swin_tiny`. DINOv2 is intentionally not silently substituted: pin the exact ViT-S/14 checkpoint used for numerical reproduction before enabling it. End-to-end ConvNeXt/Swin fine-tuning and Grad-CAM require exact historical hyperparameters/checkpoints to claim numerical reproduction.